---
title: "Private Estimation - cont — Self-study notebook"
subtitle: "Applied Data Privacy"
format:
  html:
    page-layout: full
    toc: false
    toc-depth: 2
    code-tools: true
    code-fold: true
    code-overflow: wrap
    include-in-header:
      text: |
        <style>
        .cell-output-display img, .cell-output-display .plotly-graph-div { max-width: 100%; height: auto; }
        </style>
execute:
  enabled: true
  warning: false
  message: false
jupyter: python3
---

This is the **self-study** companion to the lecture deck: full narrative + all code, meant to be
read and run. For the slide version (code hidden), open the [presentation deck](../../lecture-presentations/private-subgroup-comparisons/presentation.html).


In [ ]:
#| output: false
#| echo: false
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

try:
    import libdpy
except ImportError:
    %pip install -q "libdpy[notebook] @ git+https://github.com/applied-dp-course/pub_lib.git"
    import libdpy

%matplotlib inline


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from IPython.display import display

from libdpy.assignment_specific.private_estimation.utils import (
    DEFAULT_DELTA,
    DEFAULT_SEED,
    gaussian_noise_std,
)
from libdpy.assignment_specific.private_subgroup_comparisons.lecture_figures import (
    DEFAULT_BETA,
    DEFAULT_COUNT_SUM_AUDIT_TRIALS,
    DEFAULT_SUBGROUP_SAMPLING_DRAW_SIZE,
    DEFAULT_REPAIR_AUDIT_TRIALS,
    PTR_DELTA_FAIL,
    PTR_DELTA_RELEASE,
    PTR_DELTA_TEST,
    PTR_EPS_RELEASE,
    PTR_EPS_TEST,
    SubgroupRepairSpec,
    audit_noisy_count_sum_panel,
    audit_smooth_sensitivity_panel,
    build_oracle_ls_failure_artifact,
    build_ptr_failure_artifact,
    build_support_comparison_artifact,
    build_subgroup_sampling_artifact,
    build_fair_comparison_neighbor_pair,
    prepare_fair_comparison_frame,
    evaluate_shared_subgroup_accuracy,
    global_sensitivity_release_theory_laws,
    evaluate_subgroup_accuracy,
    make_subgroup_accuracy_leaderboard_figure,
    make_subgroup_private_methods_population_error_figure,
    make_ptr_failure_probability_figure,
    make_support_comparison_figure,
    make_subgroup_sampling_distribution_figure,
    ptr_conditional_release_theory_laws,
    # Part II — private control flow
    build_report_noisy_max_race_artifact,
    make_report_noisy_max_race_figure,
    build_report_noisy_max_accuracy_artifact,
    make_report_noisy_max_accuracy_figure,
    build_above_threshold_ladder_artifact,
    make_above_threshold_ladder_figure,
    build_guess_quality_artifact,
    make_guess_quality_figure,
)
from libdpy.assignment_specific.private_subgroup_comparisons.embed_interactives import (
    SubgroupComparisonsAuditROCVisualizer,
)
from libdpy.visualization.roc_plots import (
    TheoryROCVisualizer,
)
from libdpy.assignment_specific.private_subgroup_comparisons.mechanisms import (
    global_sensitivity_release,
    noisy_count_sum_release,
    oracle_local_sensitivity_output_law,
    ptr_support_release,
    replacement_ls_bound,
    smooth_sensitivity_release,
    subgroup_counts,
    subgroup_difference,
)
from libdpy.assignment_specific.private_subgroup_comparisons.witnesses import (
    DEFAULT_EPS_TOTAL,
    DEFAULT_SUPPORT_THRESHOLD,
    DEFAULT_TAU,
    DEFAULT_WALKTHROUGH_GROUP_COLUMN,
    DEFAULT_WALKTHROUGH_N_ROWS,
    DEFAULT_FAIR_COMPARISON_MIN_SMALL,
    DEFAULT_FAIR_COMPARISON_MAX_SMALL,
    DEFAULT_FAIR_COMPARISON_N,
    PUBLIC_CLIP_LOWER,
    PUBLIC_CLIP_UPPER,
    PUBLIC_REFERENCE_MEAN,
    PUBLIC_VALUE_UPPER,
    frame_to_arrays,
    prepare_fulton_subgroup_frame,
)

SEED = DEFAULT_SEED
EPS_TOTAL = DEFAULT_EPS_TOTAL
DELTA = DEFAULT_DELTA
rng = np.random.default_rng(SEED)

DOLLAR_SCALE = PUBLIC_REFERENCE_MEAN
frame = prepare_fair_comparison_frame(seed=SEED)
x, groups = frame_to_arrays(frame, group_column=DEFAULT_WALKTHROUGH_GROUP_COLUMN)
_, sex_groups = frame_to_arrays(frame, group_column="sex_group")

population_df = prepare_fulton_subgroup_frame(n_rows=1_000_000, seed=SEED)
population_x, population_sex_groups = frame_to_arrays(population_df)
_, population_latino_groups = frame_to_arrays(population_df, group_column="latino_group")

LEADERBOARD_N = 1000
LEADERBOARD_N_DATASETS = 40
LEADERBOARD_N_RUNS = 5


def subgroup_value_fn(df):
    return df["x"].to_numpy(dtype=float)


def subgroup_group_fn(df):
    return df[DEFAULT_WALKTHROUGH_GROUP_COLUMN].to_numpy()


# Lecture 6 — Private Subgroup Comparisons After Clipping

**Thesis:** After a private step has controlled the salary scale, subgroup analysis is about **denominators and private control flow**, not extreme outliers.

**PUBLIC CONTRACT:** We start from public quantities obtained privately in Lecture 5: a clipping interval and a reference average salary. For Fulton, use the true 5%–95% income quantile range and clipped population mean as public post-processing information:

$$
[L,U] = [q_{0.05}, q_{0.95}] = [0, 141{,}450],
\qquad
\mu_{\mathrm{ref}} = \mathbb{E}[\mathrm{clip}(Y; L, U)].
$$


## S0 — Roadmap

**Part I — one subgroup question, valid release.** Same rhythm as Lecture 5: define one statistic, inspect sampling variability, try oracle local sensitivity, audit why it fails, then repair with private control flow. After **each valid private mechanism**, run a privacy audit and read the signed-error **accuracy leaderboard** on Latino walkthrough resamples.

**Part II — private control flow.** A choice is a release too. `ReportNoisyMax` for the most common value (no public data); `AboveThreshold` for a threshold crossing made cheap by a public guess — which recovers the clip bound Part I assumed.


## S1 — The statistic

Target: difference of average mean-scaled clipped salaries between two public groups. Main demo: **Latino workers vs everyone else** (n=1,000, minority support in the 50–100 range). We also track **sex-code groups** as a stable-support contrast.


### Formal anchor: statistic and denominator floor

Fixed-size **replacement adjacency**; public denominator floor $\tau$ when counts are released.


In [ ]:
walkthrough_counts = subgroup_counts(groups)
sex_counts = subgroup_counts(sex_groups)
true_delta = subgroup_difference(x, groups)
sex_delta = subgroup_difference(x, sex_groups)
walkthrough_ls = replacement_ls_bound(
    walkthrough_counts['A'], walkthrough_counts['B'], value_range=PUBLIC_VALUE_UPPER
)
sex_ls = replacement_ls_bound(sex_counts['A'], sex_counts['B'], value_range=PUBLIC_VALUE_UPPER)
print(f"Latino walkthrough (n={DEFAULT_WALKTHROUGH_N_ROWS}): {walkthrough_counts}; LS bound {walkthrough_ls:.4f}")
print(f"Sex-code on same sample: {sex_counts}; LS bound {sex_ls:.4f}")
print(f"Latino Delta: {true_delta:.4f}; Sex-code Delta: {sex_delta:.4f}")


In [ ]:
(
    fair_x, fair_groups, fair_x_prime, fair_groups_prime,
    fair_out_idx, fair_counts, fair_counts_prime,
) = build_fair_comparison_neighbor_pair(seed=SEED)
print(f"Fair comparison D: {fair_counts} (min={min(fair_counts.values())})")
print(f"Fair comparison D': {fair_counts_prime} (min={min(fair_counts_prime.values())})")
print(f"Moved index {fair_out_idx} from smaller group {fair_groups[fair_out_idx]!r} -> {fair_groups_prime[fair_out_idx]!r}")


## S2 — Sampling variability before privacy

Before adding privacy noise, ask how much the **normalized average salary gap** moves under ordinary sampling alone. From the full Fulton population ($N \approx 25{,}766$) we draw $n=1{,}000$ records with replacement, compute the gap, and repeat 40 times. The histogram is the sampling distribution; the vertical line is the full-population target. This baseline is **not a private release** — it separates sampling error from privacy noise.


### Sex-code groups (stable ~50–50 support)

Stable denominators keep the sampling distribution relatively tight. The population sex-code gap is about $0.55$ in mean-scaled units ($\approx \$18{,}500$ after multiplying by $\mu$).


In [ ]:
sex_sampling = build_subgroup_sampling_artifact(
    population_x, population_sex_groups, sample_size=DEFAULT_SUBGROUP_SAMPLING_DRAW_SIZE, n_samples=2000,
    comparison_label='sex-code groups', seed=SEED,
)
fig = make_subgroup_sampling_distribution_figure(
    sex_sampling, title='Sampling variability — sex-code groups (Fulton)',
)
fig


### Latino workers vs everyone else (~5% minority)

The same resampling design on a rarer cut: minority denominators are smaller, so the histogram is wider even before privacy. The population gap is about $0.46$ mean-scaled units ($\approx \$15{,}400$).


In [ ]:
latino_sampling = build_subgroup_sampling_artifact(
    population_x, population_latino_groups, sample_size=DEFAULT_SUBGROUP_SAMPLING_DRAW_SIZE, n_samples=2000,
    comparison_label='Latino vs everyone else', seed=SEED,
)
fig = make_subgroup_sampling_distribution_figure(
    latino_sampling, title='Sampling variability — Latino walkthrough comparison',
)
fig


## S3 — Global sensitivity is pessimistic

If tiny groups are allowed in the domain, $GS_\Delta \le 2B$, where $B=U/\mu$. This is a valid worst-case bound for the total-function release, but it pays for datasets with nearly empty groups.

For this lecture, the point is the formula, not a plot of a reciprocal curve. The global release is valid; it is usually not the useful baseline for subgroup comparisons with decent support.


In [ ]:
gs_result = global_sensitivity_release(
    x, groups, EPS_TOTAL, rng, value_bound=PUBLIC_VALUE_UPPER, delta=DELTA,
)
print(f"Global-sensitivity noise scale: {gs_result['noise_scale']:.4f}")
print(f"Walkthrough LS / eps: {walkthrough_ls / EPS_TOTAL:.4f}")
print(f"privacy_status: {gs_result['privacy_status']}")
print(f"released Delta (mean-scaled): {gs_result['estimate']:.4f}")
print(f"released Delta (dollars): ${gs_result['estimate'] * DOLLAR_SCALE:,.0f}")


### Theory privacy audit — global sensitivity release

Global sensitivity calibrates a **fixed** Gaussian scale $2B/\varepsilon$ on every dataset. On a typical balanced neighbor pair the gap shift is tiny relative to $\sigma$, so the two output laws look almost identical — but the analytical ROC still certifies the governing $\varepsilon$ meets the claimed budget.


In [ ]:
TheoryROCVisualizer(
    distribution='Gaussian',
    scale=15.894319728207908,
    delta=0.01,
    selectable_distribution=False,
    loc_negative=0.6183934157149957,
    scale_negative=15.894319728207908,
    loc_positive=0.6106134406021159,
    scale_positive=15.894319728207908,
    compute_epsilon=True,
    show_compute_epsilon_toggle=False,
    show_governing_point=True,
).embed()


### Accuracy leaderboard — global sensitivity release

Lecture 5 signed-error decomposition on Latino walkthrough resamples.


In [ ]:
def global_mechanism(values, group_labels, run_rng):
    return global_sensitivity_release(
        values, group_labels, EPS_TOTAL, run_rng, value_bound=PUBLIC_VALUE_UPPER, delta=DELTA,
    )

global_rows = evaluate_subgroup_accuracy(
    population_df, global_mechanism, n=LEADERBOARD_N, n_datasets=LEADERBOARD_N_DATASETS,
    n_runs=LEADERBOARD_N_RUNS, seed=SEED, min_small_support=DEFAULT_FAIR_COMPARISON_MIN_SMALL, max_small_support=DEFAULT_FAIR_COMPARISON_MAX_SMALL, group_fn=subgroup_group_fn, value_fn=subgroup_value_fn,
    method='global sensitivity', privacy_status='valid', epsilon_total=EPS_TOTAL,
)
fig = make_subgroup_accuracy_leaderboard_figure(global_rows, title='Global sensitivity release')
fig


## S4 — Temptation: oracle local sensitivity

Bad proposal: Gaussian noise scaled to a **data-dependent** local-sensitivity bound — not private when hidden support moves the scale.


First on the **fair comparison Fulton pair** from S1: one replacement moves a worker **from the smaller Latino subgroup into the majority group** (salary unchanged).


The analytical Gaussian ROC uses the same fair-comparison neighbor pair with a common calibrated std (oracle-LS thought experiment on identical noise scale).


In [ ]:
TheoryROCVisualizer(
    distribution='Gaussian',
    scale=0.014609825128944782,
    delta=0.01,
    selectable_distribution=False,
    loc_negative=0.6183934157149957,
    scale_negative=0.014609825128944782,
    loc_positive=0.6106134406021159,
    scale_positive=0.014609825128944782,
    bound_epsilon=1.0,
    show_governing_point=False,
).embed()


Now move to an engineered sparse-support pair. One replacement moves the **clip-upper earner** from the 3-person minority group `B` into `A`. Support drops 3→2, so the oracle local-sensitivity bound — and therefore the Gaussian noise std — jumps on `D'`.


In [ ]:
sparse_oracle = build_oracle_ls_failure_artifact(seed=SEED)
print(
    f"D counts={sparse_oracle.counts_d}, D' counts={sparse_oracle.counts_d_prime}"
)


This is the primary privacy-failure ROC for the sparse pair. One replacement moves the **clip-upper earner** from `B` to `A` (salary unchanged), changing support 3→2 and shifting the deterministic subgroup gap by about **1.7 mean-scaled units** (sign flip). The broken oracle-LS proposal calibrates **different data-dependent Gaussian stds** on `D` and `D'`. The governing ε exceeds the claimed budget.


In [ ]:
TheoryROCVisualizer(
    distribution='Gaussian',
    scale=5.108888484066827,
    delta=0.01,
    selectable_distribution=False,
    loc_negative=-1.3266757601823869,
    scale_negative=5.108888484066827,
    loc_positive=0.34117503892853895,
    scale_positive=8.940554847116948,
    compute_epsilon=True,
    show_compute_epsilon_toggle=False,
    show_governing_point=True,
).embed()


**Diagnosis.** Local sensitivity is an excellent diagnostic, but a bad noise scale. The sparse ROC shows the output *distribution* changes between true neighbors because the mechanism's Gaussian std depends on hidden subgroup support — so this proposal is **not private** under fixed-size replacement adjacency.


## S5 — Valid repair 1: noisy counts and noisy sums

Release four Gaussian queries, then post-process with public denominator floor $\tau$. Counts are not free, but they are often the honest output.


In [ ]:
count_sum = noisy_count_sum_release(
    x, groups, EPS_TOTAL/4, EPS_TOTAL/4, EPS_TOTAL/4, EPS_TOTAL/4, DEFAULT_TAU, rng,
    value_bound=PUBLIC_VALUE_UPPER,
)
print(count_sum['privacy_status'])
print(f"Private Delta estimate (mean-scaled): {count_sum['estimate']:.4f}")
print(f"Private Delta estimate (dollars): ${count_sum['estimate'] * DOLLAR_SCALE:,.0f}")


### Empirical privacy audit — noisy counts and sums


In [ ]:
count_sum_audit = audit_noisy_count_sum_panel(
    fair_x, fair_groups, fair_x_prime, fair_groups_prime,
    eps_total=EPS_TOTAL, tau=DEFAULT_TAU, delta=DELTA,
    n_trials=DEFAULT_COUNT_SUM_AUDIT_TRIALS, seed=SEED, claimed_eps=EPS_TOTAL,
    value_bound=PUBLIC_VALUE_UPPER,
)
print(f"tau*={count_sum_audit.tau_star:.3g}, eps_plug={count_sum_audit.eps_plug:.3g}, separates={count_sum_audit.separates}")
SubgroupComparisonsAuditROCVisualizer(scene="count-sum-fair", show_compute_epsilon_toggle=True).embed()


### Accuracy leaderboard — noisy counts and sums


In [ ]:
def count_sum_mechanism(values, group_labels, run_rng):
    q = EPS_TOTAL / 4
    return noisy_count_sum_release(
        values, group_labels, q, q, q, q, DEFAULT_TAU, run_rng, value_bound=PUBLIC_VALUE_UPPER,
    )

count_sum_rows = evaluate_subgroup_accuracy(
    population_df, count_sum_mechanism, n=LEADERBOARD_N, n_datasets=LEADERBOARD_N_DATASETS,
    n_runs=LEADERBOARD_N_RUNS, seed=SEED, min_small_support=DEFAULT_FAIR_COMPARISON_MIN_SMALL, max_small_support=DEFAULT_FAIR_COMPARISON_MAX_SMALL, group_fn=subgroup_group_fn, value_fn=subgroup_value_fn,
    method='noisy counts and sums', privacy_status='valid', epsilon_total=EPS_TOTAL,
)
fig = make_subgroup_accuracy_leaderboard_figure(count_sum_rows, title='Noisy counts and sums')
fig


## S6 — Valid repair 2: Propose-Test-Release


### Formal definition: PTR support certification

Ledger: $\varepsilon=\varepsilon_{\mathrm{test}}+\varepsilon_{\mathrm{release}}$, $\delta=\delta_{\mathrm{test}}+\delta_{\mathrm{release}}+\delta_{\mathrm{fail}}$. Bad set $\mathcal B_m=\{D:\min(n_A,n_B)<m\}$; distance $d(D,\mathcal B_m)=\max\{\min(n_A,n_B)-m+1,0\}$. Buffer $b=\sigma_{\mathrm{test}}\Phi^{-1}(1-\delta_{\mathrm{fail}})$ so $P(\text{accept}\mid d=0)=\delta_{\mathrm{fail}}$.


In [ ]:
ptr = ptr_support_release(
    x, groups, DEFAULT_SUPPORT_THRESHOLD, PTR_EPS_TEST, PTR_EPS_RELEASE,
    PTR_DELTA_TEST, PTR_DELTA_RELEASE, PTR_DELTA_FAIL, rng, value_bound=PUBLIC_VALUE_UPPER,
)
print(f"status={ptr['status']!r}, accepted={ptr.get('accepted')}")
print(f"epsilon ledger: {ptr['epsilon']}")
print(f"delta ledger: {ptr['delta']}")
if ptr.get('estimate') is not None and np.isfinite(ptr['estimate']):
    print(f"released Delta (mean-scaled): {ptr['estimate']:.4f}")


### PTR test failure probability

The private support test abstains when the noisy distance falls below the public buffer. The curve shows the **analytic** probability of that test failure as true $\min(n_A,n_B)$ varies at fixed threshold $m$. The horizontal line marks the chosen failure probability $\delta$; the second line marks $P(\text{release}\mid\text{distance}=0)$ on the bad-set boundary.


In [ ]:
ptr_failure = build_ptr_failure_artifact(
    support_values=np.arange(1, 121, dtype=int),
    ptr_threshold=DEFAULT_SUPPORT_THRESHOLD, eps_test=PTR_EPS_TEST,
    delta_test=PTR_DELTA_TEST, delta_release=PTR_DELTA_RELEASE, delta_fail=PTR_DELTA_FAIL,
)
fig = make_ptr_failure_probability_figure(
    ptr_failure, title='PTR support test failure probability vs true minimum support',
    marker_min_count=min(walkthrough_counts.values()),
    marker_label=f"Latino walkthrough (min n={min(walkthrough_counts.values())})",
)
fig


### Theory privacy audit — PTR conditional release

Once the private support test accepts, PTR releases a Gaussian with **fixed** sensitivity $2B/(m-1)$. Audit with `TheoryROCVisualizer` on a neighbor pair where $\Delta(D)$ and $\Delta(D')$ separate visibly at the release scale.


In [ ]:
TheoryROCVisualizer(
    distribution='Gaussian',
    scale=0.7186256358849121,
    delta=0.016666666666666666,
    selectable_distribution=False,
    loc_negative=0.6183934157149957,
    scale_negative=0.7186256358849121,
    loc_positive=0.6106134406021159,
    scale_positive=0.7186256358849121,
    compute_epsilon=True,
    show_compute_epsilon_toggle=False,
    show_governing_point=True,
).embed()


### Accuracy leaderboard — PTR support

Conditional on release; abstentions skipped.


In [ ]:
def ptr_mechanism(values, group_labels, run_rng):
    return ptr_support_release(
        values, group_labels, DEFAULT_SUPPORT_THRESHOLD, PTR_EPS_TEST, PTR_EPS_RELEASE,
        PTR_DELTA_TEST, PTR_DELTA_RELEASE, PTR_DELTA_FAIL, run_rng, value_bound=PUBLIC_VALUE_UPPER,
    )

ptr_rows = evaluate_subgroup_accuracy(
    population_df, ptr_mechanism, n=LEADERBOARD_N, n_datasets=LEADERBOARD_N_DATASETS,
    n_runs=LEADERBOARD_N_RUNS, seed=SEED, min_small_support=DEFAULT_FAIR_COMPARISON_MIN_SMALL, max_small_support=DEFAULT_FAIR_COMPARISON_MAX_SMALL, group_fn=subgroup_group_fn, value_fn=subgroup_value_fn,
    method='PTR support', privacy_status='valid PTR', epsilon_total=EPS_TOTAL,
)
fig = make_subgroup_accuracy_leaderboard_figure(ptr_rows, title='PTR support release')
fig


## S7 — Valid repair 3: smooth sensitivity


### Formal definition: smooth sensitivity upper bound

$S^\beta(D)=\max_k e^{-\beta k}A_k(D)$. NRS Laplace release $\tilde\Delta = \Delta(D) + (2S^\beta(D)/\varepsilon)\cdot\mathrm{Lap}(1)$ when $\beta \le \varepsilon/(2\ln(2/\delta))$.


In [ ]:
smooth = smooth_sensitivity_release(
    x, groups, epsilon=EPS_TOTAL, beta=DEFAULT_BETA, rng=rng, delta=DELTA, value_bound=PUBLIC_VALUE_UPPER,
)
print(f"privacy_status: {smooth['privacy_status']}")
print(f"smooth bound: {smooth['smooth_sensitivity']:.4f}, Laplace scale: {smooth['noise_scale']:.4f}")
print(f"released Delta: {smooth['estimate']:.4f}")


### Empirical privacy audit — smooth sensitivity


In [ ]:
smooth_audit = audit_smooth_sensitivity_panel(
    fair_x, fair_groups, fair_x_prime, fair_groups_prime,
    epsilon=EPS_TOTAL, beta=DEFAULT_BETA, delta=DELTA, n_trials=DEFAULT_REPAIR_AUDIT_TRIALS,
    seed=SEED, claimed_eps=EPS_TOTAL, value_bound=PUBLIC_VALUE_UPPER,
)
print(f"eps_plug={smooth_audit.eps_plug:.3g}, separates={smooth_audit.separates}")
SubgroupComparisonsAuditROCVisualizer(scene="smooth-sensitivity-fair", show_compute_epsilon_toggle=True).embed()


### Accuracy leaderboard — smooth sensitivity


In [ ]:
def smooth_mechanism(values, group_labels, run_rng):
    return smooth_sensitivity_release(
        values, group_labels, EPS_TOTAL, DEFAULT_BETA, run_rng, delta=DELTA, value_bound=PUBLIC_VALUE_UPPER,
    )

smooth_rows = evaluate_subgroup_accuracy(
    population_df, smooth_mechanism, n=LEADERBOARD_N, n_datasets=LEADERBOARD_N_DATASETS,
    n_runs=LEADERBOARD_N_RUNS, seed=SEED, min_small_support=DEFAULT_FAIR_COMPARISON_MIN_SMALL, max_small_support=DEFAULT_FAIR_COMPARISON_MAX_SMALL, group_fn=subgroup_group_fn, value_fn=subgroup_value_fn,
    method='smooth sensitivity', privacy_status='valid smooth sensitivity', epsilon_total=EPS_TOTAL,
)
fig = make_subgroup_accuracy_leaderboard_figure(smooth_rows, title='Smooth sensitivity release')
fig


### Released estimate vs population gap — all valid private methods

Same Latino walkthrough resamples for all four mechanisms. Each curve is the distribution of **released private gaps** on the normalized salary scale; the vertical line is the full-population target (~0.46 mean-scaled units).


In [ ]:
private_method_specs = [
    SubgroupRepairSpec(global_mechanism, 'global sensitivity', 'valid', EPS_TOTAL),
    SubgroupRepairSpec(count_sum_mechanism, 'noisy counts and sums', 'valid', EPS_TOTAL),
    SubgroupRepairSpec(ptr_mechanism, 'PTR support', 'valid PTR', EPS_TOTAL),
    SubgroupRepairSpec(smooth_mechanism, 'smooth sensitivity', 'valid smooth sensitivity', EPS_TOTAL),
]
private_method_rows = evaluate_shared_subgroup_accuracy(
    population_df, private_method_specs, n=LEADERBOARD_N, n_datasets=LEADERBOARD_N_DATASETS,
    n_runs=LEADERBOARD_N_RUNS, seed=SEED, min_small_support=DEFAULT_FAIR_COMPARISON_MIN_SMALL, max_small_support=DEFAULT_FAIR_COMPARISON_MAX_SMALL, group_fn=subgroup_group_fn, value_fn=subgroup_value_fn,
)
fig = make_subgroup_private_methods_population_error_figure(
    private_method_rows,
    title='Released estimate vs population gap — valid private subgroup mechanisms',
)
fig


## S8 — Common support comparison (Part I → Part II bridge)

Fix PTR threshold $m$ and vary true minimum support $\min(n_A,n_B)$ on the same axis.

**Read the plot:** red squares are PTR abstention probability (left axis); circles/triangles are mean $|\widehat{\Delta}-\Delta|$ for count/sum, PTR (conditional on release), and smooth sensitivity (right axis). When support is tiny, PTR mostly abstains and smooth sensitivity pays large noise; as support grows, count/sum and smooth improve and PTR eventually releases. **Key detail:** PTR does not start releasing exactly at the nominal threshold — the private support test needs a buffer, so release begins above $m$, not at equality.


In [ ]:
support_artifact = build_support_comparison_artifact(
    min_support_values=np.array([3, 5, 8, 12, 20, 30, 50, 80, 120]),
    ptr_threshold=DEFAULT_SUPPORT_THRESHOLD, n_large=700, n_trials=120,
    eps_total=EPS_TOTAL, tau=DEFAULT_TAU, seed=SEED,
)
fig = make_support_comparison_figure(support_artifact)
fig


## Part II — Private control flow: choosing without releasing everything

Part I always released a **number** — a group gap with calibrated noise. But analysts also **make choices**, and a choice is a data release too: reporting a raw `argmax` or the first index over a threshold can be flipped by a single record. Part II shows two mechanisms for that, differing on one axis — **whether public data helps**:

- **Demo A — Report Noisy Max** (no public data): the most common number of years of education.
- **Demo B — AboveThreshold** (public guess): find the clipping bound $U$ that Part I *assumed* was already public.


### Demo A — Report Noisy Max: the most common value

The candidate answers (education codes 1–16) are **public** — just the column's domain. Only the counts are private. The non-private mode $\arg\max_e n_e$ can flip under one record when the top bins are near-tied, so we produce it privately: add i.i.d. noise to **every** count and return the noisy argmax. Each count has sensitivity 1 (one person sits in one bin), so this is $\varepsilon$-DP (equivalently, the exponential mechanism). Bars are true counts; dots are one noisy draw.


In [ ]:
race_artifact = build_report_noisy_max_race_artifact(seed=SEED)
fig = make_report_noisy_max_race_figure(race_artifact)
print(f"true mode = {race_artifact.nonprivate_mode}; noisy-max selected = {race_artifact.selected}")
fig


### Report Noisy Max accuracy vs. budget

Repeated draws: how often the noisy argmax recovers the true mode as $\varepsilon$ grows. A large top-bin **margin** survives noise; a near-tie does not. On this modest sample the top education bins are nearly tied, so accuracy climbs steadily with the budget — the clean privacy–utility tradeoff for unordered selection.


In [ ]:
accuracy_artifact = build_report_noisy_max_accuracy_artifact(seed=SEED, n_repeats=200)
fig = make_report_noisy_max_accuracy_figure(accuracy_artifact)
fig


### Demo B — AboveThreshold with a public guess: finding a clipping bound

Where did Part I's public clip bound $U$ come from? It is a high-salary quantile (here $p=0.95$). Public data — a national statistic, a prior year — gives a **guess** $\tilde U$. We lay a short candidate grid around it and ask AboveThreshold for the first value whose count $c_v = |\{y_i \le v\}|$ clears $T = pN$. Each $c_v$ has sensitivity 1, and AboveThreshold halts at the first crossing; the halt value is the private quantile estimate. Bars are true counts; the halted candidate is highlighted.


In [ ]:
ladder_artifact = build_above_threshold_ladder_artifact(seed=SEED, epsilon=1.0)
fig = make_above_threshold_ladder_figure(ladder_artifact)
print(
    f"private halt (clip bound) = ${ladder_artifact.halt_value:,.0f}; "
    f"true {ladder_artifact.quantile:.0%} quantile = ${ladder_artifact.true_quantile:,.0f}; "
    f"Part-I public U = ${PUBLIC_CLIP_UPPER:,.0f}"
)
fig


### A good public guess makes AboveThreshold cheap

AboveThreshold's privacy cost does not grow with the number of probed candidates, so a whole grid costs about **one query**. What matters is **where the probes land**. Below, both grids use the **same number of probes** — same privacy, same noise — a narrow grid around a good guess vs. a wide grid with no guess. The good guess spends its probes where the crossing is, so it is several times more accurate at every budget.


In [ ]:
guess_quality_artifact = build_guess_quality_artifact(seed=SEED, n_repeats=150)
fig = make_guess_quality_figure(guess_quality_artifact)
fig


### Closing the loop

The AboveThreshold halt value is exactly the kind of public clip bound $[L,U]$ that Part I assumed at the outset — a public guess, confirmed privately. Demo A returned an answer directly (the mode); Demo B produced an **input** that feeds a Part-I release. Both spent privacy on a **choice**, not on a numeric average: that is private control flow.


## Summary and bridge

We can now privately **make a choice**, not just release a number: Report Noisy Max picks the most common value with no public data, and AboveThreshold finds a threshold crossing — using a public guess to make the search cheap and closing the loop by producing the clip bound Part I assumed.

Next: private search as a general design pattern, public priors, and preparing models for private learning.
